# 6 — Ablation: Base-Rate Check

The freq=10 canary (secret `116632`) shows ~4 bits exposure even under strong DP,
while freq=50 (`791772`) is near-random (~0.7 bits). Since each frequency uses a
**different** secret, this could reflect a base-rate bias rather than memorization.

We score the 4 canary secrets against all 900k candidates on **two reference models**
to decompose the effect:

| Model | Controls for |
|-------|--------------|
| Vanilla GPT-2 (pretrained only) | — |
| GPT-2 fine-tuned on domain, no canary (`2_train.ipynb`) | domain adaptation |

If `116632` already ranks high on the domain-tuned model (before any canary exposure),
the anomaly is a prior/domain effect, not a failure of DP.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers matplotlib

## 3. Imports & Config

In [ ]:
import os
import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

CANARY_PREFIX       = 'My patient ID is '
SCORE_BATCH         = 256
MAX_SEQ_LEN         = 32
N_SECRETS           = 900_000

CANARIES_PATH       = f'{PROJECT_DIR}/data/canaries.json'
EXPOSURE_PATH       = f'{PROJECT_DIR}/results/exposure_results.json'
BASE_RATE_PATH      = f'{PROJECT_DIR}/results/base_rate.json'
BASELINE_MODEL_PATH = f'{PROJECT_DIR}/checkpoints/baseline/final'
PLOTS_DIR           = f'{PROJECT_DIR}/results/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 4. Load Canary Metadata & DP Exposure Results

In [ ]:
with open(CANARIES_PATH) as f:
    canaries = json.load(f)

with open(EXPOSURE_PATH) as f:
    exposure_results = json.load(f)

print('Canaries:')
for c in canaries:
    print(f"  freq={c['freq']:>2}x  secret={c['secret']}")

## 5. Build Candidate List

In [ ]:
all_secrets = [str(n) for n in range(100_000, 1_000_000)]
print(f'Candidate pool: {len(all_secrets):,}')

## 6. Scoring Helpers

In [ ]:
def score_candidates(model, tokenizer, prefix, candidates,
                     batch_size=SCORE_BATCH, device=device):
    """Avg per-token log-prob of (prefix + candidate) for each candidate."""
    model.eval()
    all_scores = []
    with torch.no_grad():
        for i in tqdm(range(0, len(candidates), batch_size),
                      desc='scoring', leave=False):
            batch      = candidates[i : i + batch_size]
            full_texts = [prefix + c for c in batch]
            enc = tokenizer(
                full_texts, return_tensors='pt',
                padding=True, truncation=True, max_length=MAX_SEQ_LEN,
            )
            input_ids = enc['input_ids'].to(device)
            attn_mask = enc['attention_mask'].to(device)
            outputs      = model(input_ids=input_ids, attention_mask=attn_mask)
            shift_logits = outputs.logits[:, :-1, :].float()
            shift_labels = input_ids[:, 1:]
            tok_lp = (torch.log_softmax(shift_logits, dim=-1)
                      .gather(2, shift_labels.unsqueeze(-1))
                      .squeeze(-1))
            pad_mask = (shift_labels != tokenizer.pad_token_id).float()
            n_tok    = pad_mask.sum(dim=-1).clamp(min=1)
            score    = (tok_lp * pad_mask).sum(dim=-1) / n_tok
            all_scores.extend(score.cpu().tolist())
    return np.array(all_scores)


def compute_exposure(scores, true_secret, all_secrets):
    true_idx   = all_secrets.index(true_secret)
    true_score = scores[true_idx]
    rank       = int((scores > true_score).sum()) + 1
    exposure   = math.log2(len(all_secrets)) - math.log2(rank)
    return rank, exposure, float(true_score)


print('Helpers defined.')

## 7. Score Reference Models

Score all 900 000 candidates on:
1. **Vanilla GPT-2** — pure pretraining prior
2. **Domain-tuned GPT-2** (`2_train.ipynb`, no canaries) — after adapting to the mental health domain

Each takes ~30 s on T4.

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(f'{PROJECT_DIR}/data/tokenizer')

# ── 1. Vanilla pretrained GPT-2 ───────────────────────────────────────────
print('Loading vanilla GPT-2...')
vanilla_model = GPT2LMHeadModel.from_pretrained('gpt2')
vanilla_model.resize_token_embeddings(len(tokenizer))
vanilla_model = vanilla_model.half().to(device).eval()

print('Scoring...')
vanilla_scores = score_candidates(vanilla_model, tokenizer, CANARY_PREFIX, all_secrets)
del vanilla_model
torch.cuda.empty_cache()
print('Done.')

# ── 2. Domain-tuned GPT-2 (no canaries, 2_train.ipynb) ────────────────────
print('\nLoading domain-tuned model (baseline, no canaries)...')
domain_model = GPT2LMHeadModel.from_pretrained(
    BASELINE_MODEL_PATH, torch_dtype=torch.float16).to(device).eval()

print('Scoring...')
domain_scores = score_candidates(domain_model, tokenizer, CANARY_PREFIX, all_secrets)
del domain_model
torch.cuda.empty_cache()
print('Done.')

## 8. Base-Rate Rank & Exposure for Each Canary Secret

In [ ]:
base_rate_results = []

header = (f"{'freq':>5}x  {'secret':>8}  "
          f"{'vanilla rank':>12}  {'vanilla exp':>11}  "
          f"{'domain rank':>11}  {'domain exp':>10}")
print(header)
print('-' * 75)

for c in canaries:
    v_rank, v_exp, v_lp = compute_exposure(vanilla_scores, c['secret'], all_secrets)
    d_rank, d_exp, d_lp = compute_exposure(domain_scores,  c['secret'], all_secrets)
    base_rate_results.append({
        'freq':        c['freq'],
        'secret':      c['secret'],
        'vanilla_rank': v_rank,
        'vanilla_exp':  v_exp,
        'domain_rank':  d_rank,
        'domain_exp':   d_exp,
    })
    print(f"{c['freq']:>5}x  {c['secret']:>8}  "
          f"{v_rank:>12,}  {v_exp:>11.3f}  "
          f"{d_rank:>11,}  {d_exp:>10.3f}")

with open(BASE_RATE_PATH, 'w') as f:
    json.dump(base_rate_results, f, indent=2)
print(f'\nSaved to {BASE_RATE_PATH}')

## 9. Full Comparison: Base Rates vs DP-Trained Models

If the exposure under DP barely exceeds the domain-tuned baseline,
the canary is not being memorized — it was already predictable from domain adaptation.

In [ ]:
EPS_ORDER  = [0.5, 1.0, 2.0, 4.0, 8.0, None]
EPS_LABELS = ['0.5', '1', '2', '4', '8', 'inf']

dp_lookup = {(r['epsilon_target'], r['freq']): r['exposure']
             for r in exposure_results}
br_lookup  = {r['freq']: r for r in base_rate_results}

header = (f"{'freq':>5}x  {'vanilla':>8}  {'domain':>8}  "
          + '  '.join(f'e={l:>4}' for l in EPS_LABELS))
print(header)
print('-' * 82)

for c in canaries:
    freq    = c['freq']
    br      = br_lookup[freq]
    dp_vals = [dp_lookup.get((e, freq), float('nan')) for e in EPS_ORDER]
    dp_str  = '  '.join(f'{v:>6.2f}' for v in dp_vals)
    print(f"{freq:>5}x  {br['vanilla_exp']:>8.3f}  {br['domain_exp']:>8.3f}  {dp_str}")

## 10. Plot

In [ ]:
FREQS           = [1, 5, 10, 50]
EPS_MAP         = {0.5: 0.5, 1.0: 1.0, 2.0: 2.0, 4.0: 4.0, 8.0: 8.0, None: 16.0}
EPS_TICKS       = [0.5, 1, 2, 4, 8, 16]
EPS_TICK_LABELS = ['0.5', '1', '2', '4', '8', '∞']
COLORS          = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
MARKERS         = ['o', 's', '^', 'D']

fig, ax = plt.subplots(figsize=(8, 4.5))

for freq, color, marker in zip(FREQS, COLORS, MARKERS):
    rows = sorted([r for r in exposure_results if r['freq'] == freq],
                  key=lambda r: EPS_MAP[r['epsilon_target']])
    xs   = [EPS_MAP[r['epsilon_target']] for r in rows]
    exps = [r['exposure'] for r in rows]
    ax.plot(xs, exps, marker=marker, color=color,
            linewidth=1.8, markersize=7, label=f'freq={freq}×')
    # Vanilla base-rate (open marker)
    ax.scatter([0.15], [br_lookup[freq]['vanilla_exp']], marker=marker,
               s=60, facecolors='white', edgecolors=color, linewidths=1.5, zorder=5)
    # Domain-tuned base-rate (filled marker)
    ax.scatter([0.28], [br_lookup[freq]['domain_exp']], marker=marker,
               s=60, color=color, zorder=5)

ax.axhline(0, color='grey', linewidth=1, linestyle='--', label='random')
ax.axvline(0.38, color='lightgrey', linewidth=0.8, linestyle=':')

ymax = ax.get_ylim()[1]
ax.text(0.15, ymax * 0.97, 'vanilla', ha='center', va='top', fontsize=7.5, color='grey')
ax.text(0.28, ymax * 0.97, 'domain',  ha='center', va='top', fontsize=7.5, color='grey')

ax.set_xscale('log')
ax.set_xticks(EPS_TICKS)
ax.set_xticklabels(EPS_TICK_LABELS)
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_xlabel('Privacy budget ε  (log scale)')
ax.set_ylabel('Canary exposure (bits)')
ax.set_title('Exposure vs ε — with vanilla & domain base-rates', pad=10)
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/exposure_vs_epsilon_with_baserate.png', bbox_inches='tight')
plt.show()
print('Saved.')